# Forward Implied Volatility Analysis for DAX Options

This notebook calculates **forward implied volatility** between option expiries with approximately 30 days difference.

## Methodology

1. Load DAX 40 option data from feather file
2. Calculate implied volatility for each option using Black-Scholes
3. Identify at-the-money (ATM) options for each expiry
4. Find expiry pairs with ~30 days difference (e.g., 60 vs 90 days)
5. Calculate forward volatility: $\sigma_{forward} = \sqrt{\frac{T_2 \sigma_2^2 - T_1 \sigma_1^2}{T_2 - T_1}}$

Where:
- $\sigma_1$ = implied vol for near-term expiry
- $\sigma_2$ = implied vol for far-term expiry
- $T_1$, $T_2$ = time to expiry in years

## 1. Import Libraries

In [32]:
import pandas as pd
import numpy as np
from scipy.stats import norm
from scipy.optimize import brentq
import warnings
warnings.filterwarnings('ignore')

print("✓ Libraries imported")

✓ Libraries imported


## 2. Load Options Data

In [33]:
# Load the DAX options data
df = pd.read_feather("eurex_dax40_options.feather")

print(f"Loaded {len(df):,} option contracts")
print(f"Companies: {df['underlying_isin'].nunique()}")
print(f"Expiry dates: {df['expiry_date'].nunique()}")
print(f"\nExpiry dates available:")
print(df['expiry_date'].dt.date.unique())

Loaded 34,669 option contracts
Companies: 39
Expiry dates: 16

Expiry dates available:
[datetime.date(2025, 10, 17) datetime.date(2025, 10, 24)
 datetime.date(2025, 10, 31) datetime.date(2025, 11, 7)
 datetime.date(2025, 11, 14) datetime.date(2025, 11, 21)
 datetime.date(2025, 12, 19) datetime.date(2026, 3, 20)
 datetime.date(2026, 6, 19) datetime.date(2026, 9, 18)
 datetime.date(2026, 12, 18) datetime.date(2027, 6, 18)
 datetime.date(2027, 12, 17) datetime.date(2028, 6, 16)
 datetime.date(2028, 12, 15) datetime.date(2029, 12, 21)]


## 3. Black-Scholes Functions

In [34]:
def black_scholes(S, K, T, r, sigma, option_type='call'):
    """
    Calculate Black-Scholes option price.
    
    Parameters:
    -----------
    S : float - Current stock price
    K : float - Strike price
    T : float - Time to expiration (years)
    r : float - Risk-free rate
    sigma : float - Volatility (annualized)
    option_type : str - 'call' or 'put'
    
    Returns:
    --------
    float - Option price
    """
    if T <= 0 or sigma <= 0:
        return 0
    
    d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    
    if option_type.lower() == 'call':
        price = S * norm.cdf(d1) - K * np.exp(-r * T) * norm.cdf(d2)
    else:
        price = K * np.exp(-r * T) * norm.cdf(-d2) - S * norm.cdf(-d1)
    
    return price


def implied_volatility(price, S, K, T, r, option_type='call'):
    """
    Calculate implied volatility using Brent's method.
    
    Parameters:
    -----------
    price : float - Observed option price
    S : float - Current stock price
    K : float - Strike price
    T : float - Time to expiration (years)
    r : float - Risk-free rate
    option_type : str - 'call' or 'put'
    
    Returns:
    --------
    float - Implied volatility (annualized)
    """
    if T <= 0 or price <= 0:
        return np.nan
    
    # Check for arbitrage bounds
    intrinsic = max(0, S - K) if option_type.lower() == 'call' else max(0, K - S)
    if price < intrinsic:
        return np.nan
    
    try:
        # Define function to find root
        def objective(sigma):
            return black_scholes(S, K, T, r, sigma, option_type) - price
        
        # Use Brent's method to find implied vol
        iv = brentq(objective, 0.001, 5.0, maxiter=100)
        return iv
    except:
        return np.nan


def forward_volatility(sigma1, T1, sigma2, T2):
    """
    Calculate forward implied volatility.
    
    Parameters:
    -----------
    sigma1 : float - Implied vol for near-term option
    T1 : float - Time to near-term expiry (years)
    sigma2 : float - Implied vol for far-term option
    T2 : float - Time to far-term expiry (years)
    
    Returns:
    --------
    float - Forward volatility
    """
    if T2 <= T1 or sigma1 <= 0 or sigma2 <= 0:
        return np.nan
    
    variance_term = (T2 * sigma2**2 - T1 * sigma1**2) / (T2 - T1)
    
    if variance_term < 0:
        return np.nan
    
    return np.sqrt(variance_term)


print("✓ Black-Scholes functions defined")

✓ Black-Scholes functions defined


## 4. Fetch Real Stock Prices

Using yfinance to get current stock prices for accurate calculations.

In [35]:
import yfinance as yf
from datetime import datetime, timedelta

# Mapping of ISINs to Yahoo Finance tickers
# DAX 40 companies with their Yahoo Finance symbols
ISIN_TO_TICKER = {
    'DE000A1EWWW0': 'ADS.DE',      # Adidas
    'NL0000235190': 'AIR.PA',       # Airbus (Paris)
    'DE0008404005': 'ALV.DE',       # Allianz
    'DE000BASF111': 'BAS.DE',       # BASF
    'DE000BAY0017': 'BAYN.DE',      # Bayer
    'DE0005200000': 'BEI.DE',       # Beiersdorf
    'DE0005190003': 'BMW.DE',       # BMW
    'DE000A1DAHH0': 'BNR.DE',       # Brenntag
    'DE0005439004': 'CON.DE',       # Continental
    'DE000CBK1001': 'CBK.DE',       # Commerzbank
    'DE000DTR0CK8': 'DTG.DE',       # Daimler Truck
    'DE0005140008': 'DBK.DE',       # Deutsche Bank
    'DE0005810055': 'DB1.DE',       # Deutsche Börse
    'DE0005552004': 'DHL.DE',       # Deutsche Post
    'DE0005557508': 'DTE.DE',       # Deutsche Telekom
    'DE000ENAG999': 'EOAN.DE',      # E.ON
    'DE0005785802': 'FME.DE',       # Fresenius Medical Care
    'DE0005785604': 'FRE.DE',       # Fresenius
    'DE0008402215': 'HNR1.DE',      # Hannover Rück
    'DE0006047004': 'HEI.DE',       # Heidelberg Materials
    'DE0006048432': 'HEN3.DE',      # Henkel Vz
    'DE0006231004': 'IFX.DE',       # Infineon
    'DE0007100000': 'MBG.DE',       # Mercedes-Benz
    'DE0006599905': 'MRK.DE',       # Merck
    'DE000A0D9PT0': 'MTX.DE',       # MTU Aero Engines
    'DE0008430026': 'MUV2.DE',      # Münchener Rück
    'DE000PAG9113': 'P911.DE',      # Porsche
    'DE0006969603': 'PUM.DE',       # Puma
    'NL0012169213': 'QIA.DE',       # Qiagen
    'DE0007030009': 'RHM.DE',       # Rheinmetall
    'DE0007037129': 'RWE.DE',       # RWE
    'DE0007164600': 'SAP.DE',       # SAP
    'DE0007165631': 'SRT3.DE',      # Sartorius Vz
    'DE0007236101': 'SIE.DE',       # Siemens
    'DE000ENER6Y0': 'ENR.DE',       # Siemens Energy
    'DE000SHL1006': 'SHL.DE',       # Siemens Healthineers
    'DE000SYM9999': 'SY1.DE',       # Symrise
    'DE0007664039': 'VOW3.DE',      # Volkswagen Vz
    'DE000A1ML7J1': 'VNA.DE',       # Vonovia
    'DE000ZAL1111': 'ZAL.DE',       # Zalando
}

def fetch_real_prices(isin_list, max_retries=3):
    """
    Fetch real current stock prices using yfinance.
    
    Parameters:
    -----------
    isin_list : list - List of ISINs
    max_retries : int - Number of retry attempts for failed fetches
    
    Returns:
    --------
    dict - Dictionary mapping ISIN to current price
    """
    prices = {}
    failed = []
    
    print(f"Fetching real-time prices for {len(isin_list)} stocks...")
    print("This may take a moment...\n")
    
    for isin in isin_list:
        ticker_symbol = ISIN_TO_TICKER.get(isin)
        
        if ticker_symbol is None:
            print(f"  ⚠️  {isin}: No ticker mapping found")
            failed.append(isin)
            continue
        
        try:
            ticker = yf.Ticker(ticker_symbol)
            
            # Try to get current price from multiple sources
            price = None
            
            # Method 1: Current price from info
            try:
                info = ticker.info
                price = info.get('currentPrice') or info.get('regularMarketPrice') or info.get('previousClose')
            except:
                pass
            
            # Method 2: Latest close from history
            if price is None:
                try:
                    hist = ticker.history(period='5d')
                    if len(hist) > 0:
                        price = hist['Close'].iloc[-1]
                except:
                    pass
            
            if price is not None and price > 0:
                prices[isin] = float(price)
                # Get company name for display
                company_name = df[df['underlying_isin'] == isin].iloc[0]['name'].split('(')[0].strip()
                print(f"  ✓ {company_name[:30]:<30} ({ticker_symbol:>8}): €{price:>8.2f}")
            else:
                print(f"  ⚠️  {ticker_symbol}: Could not fetch price")
                failed.append(isin)
                
        except Exception as e:
            print(f"  ❌ {ticker_symbol}: Error - {str(e)[:50]}")
            failed.append(isin)
    
    print(f"\n✓ Successfully fetched prices for {len(prices)}/{len(isin_list)} stocks")
    
    if len(failed) > 0:
        print(f"⚠️  Failed to fetch {len(failed)} stocks: {failed[:5]}...")
    
    return prices

# Fetch real prices for all stocks in our dataset
unique_isins = df['underlying_isin'].unique()
spot_prices = fetch_real_prices(unique_isins)

print(f"\n{'='*70}")
print(f"REAL PRICE DATA SUMMARY")
print(f"{'='*70}")
print(f"Total stocks in dataset: {len(unique_isins)}")
print(f"Prices fetched: {len(spot_prices)}")
print(f"Success rate: {len(spot_prices)/len(unique_isins)*100:.1f}%")
print(f"{'='*70}")

Fetching real-time prices for 39 stocks...
This may take a moment...

  ✓ Deutsche Bank AG               (  DBK.DE): €   30.27
  ✓ Deutsche Bank AG               (  DBK.DE): €   30.27
  ✓ Bayerische Motoren Werke AG    (  BMW.DE): €   78.92
  ✓ Bayerische Motoren Werke AG    (  BMW.DE): €   78.92
  ✓ Beiersdorf AG                  (  BEI.DE): €   91.24
  ✓ Beiersdorf AG                  (  BEI.DE): €   91.24
  ✓ Continental AG                 (  CON.DE): €   54.48
  ✓ Continental AG                 (  CON.DE): €   54.48
  ✓ Deutsche Post AG               (  DHL.DE): €   38.66
  ✓ Deutsche Post AG               (  DHL.DE): €   38.66
  ✓ Deutsche Telekom AG            (  DTE.DE): €   29.75
  ✓ Deutsche Telekom AG            (  DTE.DE): €   29.75
  ✓ Fresenius SE & Co. KGaA        (  FRE.DE): €   48.60
  ✓ Fresenius SE & Co. KGaA        (  FRE.DE): €   48.60
  ✓ Fresenius Medical Care AG      (  FME.DE): €   47.43
  ✓ Fresenius Medical Care AG      (  FME.DE): €   47.43
  ✓ Deutsche Börse

## 5. Calculate Implied Volatility for All Options

This will take a few minutes as we calculate IV for ~35,000 options.

In [36]:
# Risk-free rate assumption (ECB rate)
RISK_FREE_RATE = 0.03  # 3%

# Add spot prices to dataframe
df['spot_price'] = df['underlying_isin'].map(spot_prices)

# Filter out options without spot price estimate
df = df[df['spot_price'].notna()].copy()

# Calculate time to expiry in years
df['T'] = df['days_to_expiry'] / 365.0

# Calculate moneyness (S/K)
df['moneyness'] = df['spot_price'] / df['strikePrice']

# Filter out deeply ITM/OTM options (focus on reasonable strikes)
df = df[(df['moneyness'] > 0.7) & (df['moneyness'] < 1.3)].copy()

# Filter out options with very low prices or zero
df = df[df['settlementPrice'] > 0.01].copy()

# Filter out options expiring too soon (< 5 days) or too far (> 2 years)
df = df[(df['days_to_expiry'] >= 5) & (df['days_to_expiry'] <= 730)].copy()

print(f"Calculating implied volatility for {len(df):,} options...")
print("This may take a few minutes...\n")

# Calculate implied volatility
df['implied_vol'] = df.apply(
    lambda row: implied_volatility(
        price=row['settlementPrice'],
        S=row['spot_price'],
        K=row['strikePrice'],
        T=row['T'],
        r=RISK_FREE_RATE,
        option_type=row['option_type'].lower()
    ),
    axis=1
)

# Filter out failed calculations
df_with_iv = df[df['implied_vol'].notna()].copy()

# Filter out unrealistic volatilities (< 5% or > 200%)
df_with_iv = df_with_iv[
    (df_with_iv['implied_vol'] > 0.05) & 
    (df_with_iv['implied_vol'] < 2.0)
].copy()

print(f"✓ Calculated IV for {len(df_with_iv):,} options")
print(f"  Failed/filtered: {len(df) - len(df_with_iv):,}")
print(f"\nIV Statistics:")
print(f"  Mean: {df_with_iv['implied_vol'].mean():.2%}")
print(f"  Median: {df_with_iv['implied_vol'].median():.2%}")
print(f"  Min: {df_with_iv['implied_vol'].min():.2%}")
print(f"  Max: {df_with_iv['implied_vol'].max():.2%}")

Calculating implied volatility for 14,851 options...
This may take a few minutes...

✓ Calculated IV for 13,178 options
  Failed/filtered: 1,673

IV Statistics:
  Mean: 38.74%
  Median: 36.29%
  Min: 5.47%
  Max: 167.47%
✓ Calculated IV for 13,178 options
  Failed/filtered: 1,673

IV Statistics:
  Mean: 38.74%
  Median: 36.29%
  Min: 5.47%
  Max: 167.47%


## 6. Identify At-The-Money (ATM) Options

For each stock and expiry, find the strike closest to the spot price.

In [37]:
def find_atm_options(df_group):
    """
    Find ATM call and put for a given stock/expiry combination.
    ATM = strike closest to spot price.
    """
    if len(df_group) == 0:
        return pd.DataFrame()
    
    spot = df_group['spot_price'].iloc[0]
    
    # Find strike closest to spot
    df_group = df_group.copy()
    df_group['distance_to_atm'] = abs(df_group['strikePrice'] - spot)
    
    results = []
    
    # Get ATM call
    calls = df_group[df_group['option_type'] == 'CALL']
    if len(calls) > 0:
        atm_call = calls.loc[calls['distance_to_atm'].idxmin()]
        results.append(atm_call)
    
    # Get ATM put
    puts = df_group[df_group['option_type'] == 'PUT']
    if len(puts) > 0:
        atm_put = puts.loc[puts['distance_to_atm'].idxmin()]
        results.append(atm_put)
    
    if len(results) == 0:
        return pd.DataFrame()
    
    return pd.DataFrame(results)


# Find ATM options for each stock/expiry combination
atm_options = df_with_iv.groupby(
    ['underlying_isin', 'expiry_date'],
    group_keys=False
).apply(find_atm_options).reset_index(drop=True)

print(f"Found {len(atm_options):,} ATM options")
print(f"Stocks: {atm_options['underlying_isin'].nunique()}")
print(f"Expiry dates: {atm_options['expiry_date'].nunique()}")

Found 746 ATM options
Stocks: 39
Expiry dates: 11


## 7. Find Expiry Pairs with ~30 Days Difference

In [38]:
def find_expiry_pairs(expiry_dates, target_diff=30, tolerance=7):
    """
    Find pairs of expiry dates with approximately target_diff days between them.
    
    Parameters:
    -----------
    expiry_dates : list - List of expiry dates
    target_diff : int - Target difference in days (default 30)
    tolerance : int - Acceptable difference tolerance (default ±7 days)
    
    Returns:
    --------
    list of tuples - (near_expiry, far_expiry, actual_diff)
    """
    expiry_dates = sorted(expiry_dates)
    pairs = []
    
    for i, exp1 in enumerate(expiry_dates):
        for exp2 in expiry_dates[i+1:]:
            diff = (exp2 - exp1).days
            if abs(diff - target_diff) <= tolerance:
                pairs.append((exp1, exp2, diff))
    
    return pairs


# Get unique expiry dates
unique_expiries = sorted(atm_options['expiry_date'].unique())

print(f"Available expiry dates: {len(unique_expiries)}")
print(f"\nExpiry dates:")
for exp in unique_expiries:
    days = (exp - pd.Timestamp.now(tz='UTC')).days
    print(f"  {exp.date()} ({days} days)")

# Find pairs
expiry_pairs = find_expiry_pairs(unique_expiries, target_diff=30, tolerance=7)

print(f"\n\nFound {len(expiry_pairs)} expiry pairs with ~30 days difference:")
print("="*70)
for near, far, diff in expiry_pairs:
    near_days = (near - pd.Timestamp.now(tz='UTC')).days
    far_days = (far - pd.Timestamp.now(tz='UTC')).days
    print(f"{near.date()} ({near_days:3}d) → {far.date()} ({far_days:3}d) : {diff} days apart")

Available expiry dates: 11

Expiry dates:
  2025-10-24 (9 days)
  2025-10-31 (16 days)
  2025-11-07 (23 days)
  2025-11-14 (30 days)
  2025-11-21 (37 days)
  2025-12-19 (65 days)
  2026-03-20 (156 days)
  2026-06-19 (247 days)
  2026-09-18 (338 days)
  2026-12-18 (429 days)
  2027-06-18 (611 days)


Found 3 expiry pairs with ~30 days difference:
2025-10-24 (  9d) → 2025-11-21 ( 37d) : 28 days apart
2025-11-14 ( 30d) → 2025-12-19 ( 65d) : 35 days apart
2025-11-21 ( 37d) → 2025-12-19 ( 65d) : 28 days apart


## 8. Calculate Forward Implied Volatility

For each stock and expiry pair, calculate the forward volatility.

In [39]:
forward_vols = []

for near_exp, far_exp, days_diff in expiry_pairs:
    # For each stock
    for isin in atm_options['underlying_isin'].unique():
        # Get ATM options for near expiry
        near_opts = atm_options[
            (atm_options['underlying_isin'] == isin) &
            (atm_options['expiry_date'] == near_exp)
        ]
        
        # Get ATM options for far expiry
        far_opts = atm_options[
            (atm_options['underlying_isin'] == isin) &
            (atm_options['expiry_date'] == far_exp)
        ]
        
        if len(near_opts) == 0 or len(far_opts) == 0:
            continue
        
        # Calculate average IV for near expiry (call + put)
        near_iv = near_opts['implied_vol'].mean()
        near_T = near_opts['T'].mean()
        near_strike = near_opts['strikePrice'].mean()
        
        # Calculate average IV for far expiry (call + put)
        far_iv = far_opts['implied_vol'].mean()
        far_T = far_opts['T'].mean()
        far_strike = far_opts['strikePrice'].mean()
        
        # Calculate forward volatility
        fwd_vol = forward_volatility(near_iv, near_T, far_iv, far_T)
        
        if not np.isnan(fwd_vol):
            forward_vols.append({
                'underlying_isin': isin,
                'near_expiry': near_exp,
                'far_expiry': far_exp,
                'days_between': days_diff,
                'near_days_to_exp': near_opts['days_to_expiry'].mean(),
                'far_days_to_exp': far_opts['days_to_expiry'].mean(),
                'near_strike': near_strike,
                'far_strike': far_strike,
                'spot_price': near_opts['spot_price'].iloc[0],
                'near_T': near_T,
                'far_T': far_T,
                'near_iv': near_iv,
                'far_iv': far_iv,
                'forward_vol': fwd_vol,
                'near_open_interest': near_opts['openInterest'].sum(),
                'far_open_interest': far_opts['openInterest'].sum()
            })

# Create DataFrame
df_forward = pd.DataFrame(forward_vols)

print(f"✓ Calculated forward volatility for {len(df_forward)} stock/expiry combinations")
print(f"\nForward Vol Statistics:")
print(f"  Mean: {df_forward['forward_vol'].mean():.2%}")
print(f"  Median: {df_forward['forward_vol'].median():.2%}")
print(f"  Std Dev: {df_forward['forward_vol'].std():.2%}")
print(f"  Min: {df_forward['forward_vol'].min():.2%}")
print(f"  Max: {df_forward['forward_vol'].max():.2%}")

✓ Calculated forward volatility for 89 stock/expiry combinations

Forward Vol Statistics:
  Mean: 30.18%
  Median: 28.20%
  Std Dev: 8.78%
  Min: 17.94%
  Max: 61.14%


## 9. Results: Forward Volatility by Stock and Expiry Pair

In [40]:
# Load stock names for display
import json

# Create ISIN to name mapping from the original data
isin_to_name = {}
df_orig = pd.read_feather("eurex_dax40_options.feather")
for isin in df_orig['underlying_isin'].unique():
    # Extract company name from option name
    sample = df_orig[df_orig['underlying_isin'] == isin].iloc[0]['name']
    company_name = sample.split('(')[0].strip()
    isin_to_name[isin] = company_name

df_forward['company_name'] = df_forward['underlying_isin'].map(isin_to_name)

# Sort by forward volatility
df_display = df_forward.sort_values('forward_vol', ascending=False).copy()

print("="*100)
print(f"{'FORWARD IMPLIED VOLATILITY ANALYSIS':^100}")
print("="*100)
print(f"\n{'Company':<30} {'Near→Far':<25} {'Near IV':>10} {'Far IV':>10} {'Fwd Vol':>10} {'OI':<15}")
print("-"*100)

for idx, row in df_display.iterrows():
    company = row['company_name'][:28] if len(row['company_name']) > 28 else row['company_name']
    exp_pair = f"{row['near_expiry'].strftime('%Y-%m-%d')}→{row['far_expiry'].strftime('%m-%d')}"
    oi = f"{row['near_open_interest']:.0f}/{row['far_open_interest']:.0f}"
    
    print(f"{company:<30} {exp_pair:<25} {row['near_iv']:>9.1%} {row['far_iv']:>9.1%} {row['forward_vol']:>9.1%} {oi:<15}")

print("\n" + "="*100)

                                FORWARD IMPLIED VOLATILITY ANALYSIS                                 

Company                        Near→Far                     Near IV     Far IV    Fwd Vol OI             
----------------------------------------------------------------------------------------------------
Siemens Energy AG              2025-10-24→11-21              60.9%     61.1%     61.1% 1/197          
Bayer AG                       2025-10-24→11-21              45.7%     51.6%     53.6% 58/3671        
Rheinmetall AG                 2025-10-24→11-21              52.3%     52.4%     52.4% 2/74           
Siemens Energy AG              2025-11-14→12-19              62.8%     55.6%     48.5% 0/1204         
Bayer AG                       2025-11-21→12-19              51.6%     50.1%     47.9% 3671/975       
Siemens Energy AG              2025-11-21→12-19              61.1%     55.6%     47.3% 197/1204       
Bayer AG                       2025-11-14→12-19              53.4%     50

## 10. Summary by Expiry Pair

In [41]:
# Group by expiry pair
summary = df_forward.groupby(['near_expiry', 'far_expiry']).agg({
    'forward_vol': ['mean', 'median', 'std', 'min', 'max', 'count'],
    'near_iv': 'mean',
    'far_iv': 'mean'
})

print("="*90)
print(f"{'FORWARD VOLATILITY BY EXPIRY PAIR':^90}")
print("="*90)
print(f"\n{'Near Expiry':<12} {'Far Expiry':<12} {'Stocks':>7} {'Mean Fwd':>10} {'Median':>10} {'Std':>10} {'Range':>15}")
print("-"*90)

for (near, far), row in summary.iterrows():
    mean_fwd = row[('forward_vol', 'mean')]
    median_fwd = row[('forward_vol', 'median')]
    std_fwd = row[('forward_vol', 'std')]
    min_fwd = row[('forward_vol', 'min')]
    max_fwd = row[('forward_vol', 'max')]
    count = int(row[('forward_vol', 'count')])
    range_str = f"{min_fwd:.1%}-{max_fwd:.1%}"
    
    print(f"{near.date()} {far.date()} {count:>7} {mean_fwd:>9.1%} {median_fwd:>9.1%} {std_fwd:>9.1%} {range_str:>15}")

print("\n" + "="*90)

                            FORWARD VOLATILITY BY EXPIRY PAIR                             

Near Expiry  Far Expiry    Stocks   Mean Fwd     Median        Std           Range
------------------------------------------------------------------------------------------
2025-10-24 2025-11-21      25     33.1%     30.1%     10.6%     20.6%-61.1%
2025-11-14 2025-12-19      25     29.4%     27.1%      8.1%     18.1%-48.5%
2025-11-21 2025-12-19      39     28.8%     27.3%      7.6%     17.9%-47.9%



## 11. Top 20 Highest Forward Volatilities

## 11. High Forward Volatility Ratio Analysis

The forward volatility ratio compares the forward implied volatility to the near-term spot volatility:
- **Ratio > 1.0**: Market expects volatility to increase (event risk ahead)
- **Ratio < 1.0**: Market expects volatility to decrease (calm period expected)
- **Ratio ≈ 1.0**: Flat term structure (no major expectations change)

In [42]:
# Calculate forward volatility ratio
df_forward['fwd_vol_ratio'] = df_forward['forward_vol'] / df_forward['near_iv']

# Sort by ratio to find stocks with highest forward vol expectations
df_by_ratio = df_forward.sort_values('fwd_vol_ratio', ascending=False).copy()

print("="*120)
print(f"{'STOCKS WITH HIGHEST FORWARD VOLATILITY RATIO':^120}")
print("="*120)
print(f"\n{'Rank':<6} {'Company':<30} {'Near→Far Expiry':<23} {'Near IV':>10} {'Fwd Vol':>10} {'Ratio':>10} {'Signal':<20}")
print("-"*120)

for rank, (idx, row) in enumerate(df_by_ratio.head(30).iterrows(), 1):
    company = row['company_name'][:28]
    exp_pair = f"{row['near_expiry'].strftime('%Y-%m-%d')}→{row['far_expiry'].strftime('%m-%d')}"
    ratio = row['fwd_vol_ratio']
    
    # Interpretation signal
    if ratio > 1.3:
        signal = "🔴 Strong increase"
    elif ratio > 1.1:
        signal = "🟡 Moderate increase"
    elif ratio > 0.9:
        signal = "🟢 Flat/stable"
    else:
        signal = "🔵 Decrease expected"
    
    print(f"{rank:<6} {company:<30} {exp_pair:<23} {row['near_iv']:>9.1%} {row['forward_vol']:>9.1%} {ratio:>9.2f} {signal:<20}")

print("\n" + "="*120)

# Summary statistics
print("\n" + "="*120)
print(f"{'FORWARD VOLATILITY RATIO STATISTICS':^120}")
print("="*120)
print(f"\nMean ratio: {df_forward['fwd_vol_ratio'].mean():.2f}")
print(f"Median ratio: {df_forward['fwd_vol_ratio'].median():.2f}")
print(f"Std dev: {df_forward['fwd_vol_ratio'].std():.2f}")
print(f"\nStocks expecting vol increase (ratio > 1.1): {len(df_forward[df_forward['fwd_vol_ratio'] > 1.1])}")
print(f"Stocks expecting vol stable (0.9 < ratio < 1.1): {len(df_forward[(df_forward['fwd_vol_ratio'] >= 0.9) & (df_forward['fwd_vol_ratio'] <= 1.1)])}")
print(f"Stocks expecting vol decrease (ratio < 0.9): {len(df_forward[df_forward['fwd_vol_ratio'] < 0.9])}")
print("="*120)

                                      STOCKS WITH HIGHEST FORWARD VOLATILITY RATIO                                      

Rank   Company                        Near→Far Expiry            Near IV    Fwd Vol      Ratio Signal              
------------------------------------------------------------------------------------------------------------------------
1      Bayer AG                       2025-10-24→11-21            45.7%     53.6%      1.17 🟡 Moderate increase 
2      Allianz SE                     2025-10-24→11-21            18.5%     20.6%      1.11 🟡 Moderate increase 
3      Deutsche Post AG               2025-10-24→11-21            24.5%     27.2%      1.11 🟡 Moderate increase 
4      Allianz SE                     2025-11-14→12-19            19.0%     21.1%      1.11 🟡 Moderate increase 
5      Siemens AG                     2025-10-24→11-21            35.4%     38.1%      1.07 🟢 Flat/stable       
6      Deutsche Bank AG               2025-10-24→11-21            35.2%     

## 12. Top Stocks by Expiry Period

In [43]:
# Show top stocks for each expiry pair
print("="*120)
print(f"{'TOP 5 STOCKS WITH HIGHEST FORWARD VOL RATIO BY EXPIRY PAIR':^120}")
print("="*120)

for near_exp, far_exp, days_diff in expiry_pairs:
    # Filter for this expiry pair
    df_pair = df_by_ratio[
        (df_by_ratio['near_expiry'] == near_exp) & 
        (df_by_ratio['far_expiry'] == far_exp)
    ].head(5)
    
    if len(df_pair) > 0:
        print(f"\n{near_exp.strftime('%Y-%m-%d')} → {far_exp.strftime('%Y-%m-%d')} ({days_diff} days apart)")
        print("-"*120)
        print(f"{'Company':<30} {'Spot €':>10} {'Near IV':>10} {'Far IV':>10} {'Fwd Vol':>10} {'Ratio':>10} {'Signal':<20}")
        print("-"*120)
        
        for idx, row in df_pair.iterrows():
            company = row['company_name'][:28]
            ratio = row['fwd_vol_ratio']
            
            if ratio > 1.3:
                signal = "🔴 Strong increase"
            elif ratio > 1.1:
                signal = "🟡 Moderate increase"
            elif ratio > 0.9:
                signal = "🟢 Flat/stable"
            else:
                signal = "🔵 Decrease expected"
            
            print(f"{company:<30} {row['spot_price']:>10.2f} {row['near_iv']:>9.1%} {row['far_iv']:>9.1%} {row['forward_vol']:>9.1%} {ratio:>9.2f} {signal:<20}")

print("\n" + "="*120)

                               TOP 5 STOCKS WITH HIGHEST FORWARD VOL RATIO BY EXPIRY PAIR                               

2025-10-24 → 2025-11-21 (28 days apart)
------------------------------------------------------------------------------------------------------------------------
Company                            Spot €    Near IV     Far IV    Fwd Vol      Ratio Signal              
------------------------------------------------------------------------------------------------------------------------
Bayer AG                            27.75     45.7%     51.6%     53.6%      1.17 🟡 Moderate increase 
Allianz SE                         368.90     18.5%     20.1%     20.6%      1.11 🟡 Moderate increase 
Deutsche Post AG                    38.66     24.5%     26.5%     27.2%      1.11 🟡 Moderate increase 
Siemens AG                         238.85     35.4%     37.4%     38.1%      1.07 🟢 Flat/stable       
Deutsche Bank AG                    30.27     35.2%     36.5%     36.9%      

## 13. Calendar Spread Opportunity Analysis

Calendar spreads profit when:
1. **Forward vol ratio > 1.0**: Far options relatively cheap vs near options
2. **Stock stays near ATM strike**: Maximum theta decay benefit
3. **Near-term vol stays low**: Short leg loses value faster
4. **Far-term vol increases**: Long leg gains value

**Risk factors:**
- Early volatility spike hurts short leg
- Large stock moves reduce profitability
- Event risk (earnings, regulatory) timing is critical

In [44]:
# Calendar Spread Scoring System
# Score based on: FV ratio, absolute vol levels, moneyness, open interest

def score_calendar_spread(row):
    """
    Score a calendar spread opportunity (0-100)
    Higher score = better opportunity
    """
    score = 0
    
    # Factor 1: Forward Vol Ratio (40 points max)
    # Ideal: 1.1 to 1.3 (too high = event risk)
    ratio = row['fwd_vol_ratio']
    if ratio > 1.3:
        score += 25  # High risk, high reward
    elif ratio > 1.15:
        score += 40  # Sweet spot
    elif ratio > 1.05:
        score += 30
    else:
        score += 10
    
    # Factor 2: Near-term IV level (20 points max)
    # Prefer moderate levels (20-40%)
    near_iv = row['near_iv']
    if 0.20 <= near_iv <= 0.40:
        score += 20  # Ideal range
    elif 0.15 <= near_iv <= 0.50:
        score += 15
    else:
        score += 5
    
    # Factor 3: Open Interest (20 points max)
    # Higher OI = better liquidity
    total_oi = row['near_open_interest'] + row['far_open_interest']
    if total_oi > 10000:
        score += 20
    elif total_oi > 5000:
        score += 15
    elif total_oi > 1000:
        score += 10
    else:
        score += 5
    
    # Factor 4: Days to near expiry (20 points max)
    # Prefer 20-40 days for optimal time decay
    days = row['near_days_to_exp']
    if 20 <= days <= 40:
        score += 20
    elif 15 <= days <= 50:
        score += 15
    elif 10 <= days <= 60:
        score += 10
    else:
        score += 5
    
    return score

# Calculate scores for all opportunities
df_forward['calendar_score'] = df_forward.apply(score_calendar_spread, axis=1)

# Filter for high FV ratio only (potential calendar spread candidates)
df_calendar = df_forward[df_forward['fwd_vol_ratio'] > 1.05].copy()
df_calendar = df_calendar.sort_values('calendar_score', ascending=False)

print("="*130)
print(f"{'CALENDAR SPREAD OPPORTUNITIES (Sorted by Score)':^130}")
print("="*130)
print(f"\n{'Score':<7} {'Company':<28} {'Near→Far':<20} {'Ratio':>7} {'Near IV':>9} {'Fwd Vol':>9} {'Days':>6} {'Total OI':>10} {'Grade':<12}")
print("-"*130)

for idx, row in df_calendar.head(20).iterrows():
    company = row['company_name'][:26]
    exp_pair = f"{row['near_expiry'].strftime('%m/%d')}→{row['far_expiry'].strftime('%m/%d')}"
    score = row['calendar_score']
    total_oi = row['near_open_interest'] + row['far_open_interest']
    
    # Grade the opportunity
    if score >= 80:
        grade = "🟢 Excellent"
    elif score >= 70:
        grade = "🟡 Good"
    elif score >= 60:
        grade = "🟠 Fair"
    else:
        grade = "🔴 Risky"
    
    print(f"{score:<7.0f} {company:<28} {exp_pair:<20} {row['fwd_vol_ratio']:>6.2f} {row['near_iv']:>8.1%} {row['forward_vol']:>8.1%} {row['near_days_to_exp']:>5.0f} {total_oi:>10.0f} {grade:<12}")

print("\n" + "="*130)

# Summary by grade
print("\n" + "="*130)
print(f"{'OPPORTUNITY SUMMARY':^130}")
print("="*130)
excellent = len(df_calendar[df_calendar['calendar_score'] >= 80])
good = len(df_calendar[(df_calendar['calendar_score'] >= 70) & (df_calendar['calendar_score'] < 80)])
fair = len(df_calendar[(df_calendar['calendar_score'] >= 60) & (df_calendar['calendar_score'] < 70)])
risky = len(df_calendar[df_calendar['calendar_score'] < 60])

print(f"\n🟢 Excellent opportunities (score ≥ 80): {excellent}")
print(f"🟡 Good opportunities (score 70-79): {good}")
print(f"🟠 Fair opportunities (score 60-69): {fair}")
print(f"🔴 Risky opportunities (score < 60): {risky}")
print(f"\nTotal candidates with FV ratio > 1.05: {len(df_calendar)}")
print("="*130)

                                         CALENDAR SPREAD OPPORTUNITIES (Sorted by Score)                                          

Score   Company                      Near→Far               Ratio   Near IV   Fwd Vol   Days   Total OI Grade       
----------------------------------------------------------------------------------------------------------------------------------
75      Bayer AG                     10/24→11/21            1.17    45.7%    53.6%    10       3729 🟡 Good      
75      Allianz SE                   11/14→12/19            1.11    19.0%    21.1%    31       1732 🟡 Good      
70      Deutsche Bank AG             10/24→11/21            1.05    35.2%    36.9%    10       1161 🟡 Good      
65      Deutsche Post AG             10/24→11/21            1.11    24.5%    27.2%    10        938 🟠 Fair      
65      Siemens AG                   10/24→11/21            1.07    35.4%    38.1%    10        703 🟠 Fair      
60      Allianz SE                   10/24→11/21       

In [45]:
top_20 = df_display.head(20)

print("="*110)
print(f"{'TOP 20 HIGHEST FORWARD IMPLIED VOLATILITIES':^110}")
print("="*110)
print(f"\n{'Rank':<6} {'Company':<30} {'Near→Far Expiry':<23} {'Near IV':>10} {'Far IV':>10} {'Fwd Vol':>10}")
print("-"*110)

for rank, (idx, row) in enumerate(top_20.iterrows(), 1):
    company = row['company_name'][:28]
    exp_pair = f"{row['near_expiry'].strftime('%Y-%m-%d')}→{row['far_expiry'].strftime('%m-%d')}"
    
    print(f"{rank:<6} {company:<30} {exp_pair:<23} {row['near_iv']:>9.1%} {row['far_iv']:>9.1%} {row['forward_vol']:>9.1%}")

print("\n" + "="*110)

                                 TOP 20 HIGHEST FORWARD IMPLIED VOLATILITIES                                  

Rank   Company                        Near→Far Expiry            Near IV     Far IV    Fwd Vol
--------------------------------------------------------------------------------------------------------------
1      Siemens Energy AG              2025-10-24→11-21            60.9%     61.1%     61.1%
2      Bayer AG                       2025-10-24→11-21            45.7%     51.6%     53.6%
3      Rheinmetall AG                 2025-10-24→11-21            52.3%     52.4%     52.4%
4      Siemens Energy AG              2025-11-14→12-19            62.8%     55.6%     48.5%
5      Bayer AG                       2025-11-21→12-19            51.6%     50.1%     47.9%
6      Siemens Energy AG              2025-11-21→12-19            61.1%     55.6%     47.3%
7      Bayer AG                       2025-11-14→12-19            53.4%     50.1%     46.9%
8      Rheinmetall AG                 

## 12. Save Results

In [46]:
# Save to CSV for further analysis
output_file = "forward_volatility_results.csv"
df_forward.to_csv(output_file, index=False)

print(f"✓ Results saved to {output_file}")
print(f"  Rows: {len(df_forward):,}")
print(f"  Columns: {len(df_forward.columns)}")

# Also save to feather for fast loading
feather_file = "forward_volatility_results.feather"
df_forward.to_feather(feather_file)
print(f"\n✓ Also saved to {feather_file} (faster loading)")

✓ Results saved to forward_volatility_results.csv
  Rows: 89
  Columns: 19

✓ Also saved to forward_volatility_results.feather (faster loading)


## 13. Interpretation

### What is Forward Implied Volatility?

Forward implied volatility represents the market's expectation of volatility **between two future dates**, not from today.

For example, if we have:
- 60-day option with IV = 30%
- 90-day option with IV = 35%

The forward volatility tells us what volatility the market expects **from day 60 to day 90** (the 30-day period between the two expiries).

### Why is this useful?

1. **Event Risk**: High forward vol may indicate expected events (earnings, policy decisions)
2. **Term Structure**: Shows how volatility expectations change over time
3. **Trading Opportunities**: Calendar spreads can profit from forward vol mispricing
4. **Risk Management**: Better understanding of future volatility expectations

### Interpretation:

- **Forward Vol > Spot Vol**: Market expects volatility to increase
- **Forward Vol < Spot Vol**: Market expects volatility to decrease
- **High Forward Vol**: Potential event risk in that period
- **Low Forward Vol**: Market expects calm period